# पाठ 10 - उत्पादनमा AI एजेन्टहरू

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरेर AI एजेन्टहरूका लागि **उत्पादन ढाँचा** सिक्नुहुनेछ। हामीले समेट्छौं:

- **प्रवेक्षणयोग्यता** — एजेन्ट अन्तरक्रियाहरूमा समय मापन र लगिङ थप्ने
- **मूल्याङ्कन** — प्रतिक्रिया गुणस्तर स्कोर गर्न एक मूल्याङ्कनकर्ता एजेन्ट प्रयोग गर्ने
- **खर्च व्यवस्थापन** — टोकन अनुकूलन र मोडेल छनोटका रणनीतिहरू

परिदृश्य एक **यात्रा एजेन्ट** हो जसले प्रयोगकर्ताहरूलाई यात्रा योजना बनाउन मद्दत गर्छ, र त्यसमा अनुगमन तथा मूल्याङ्कन तहहरू थपिएका छन्।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## उत्पादन विचारहरू

प्रोटोटाइपबाट उत्पादनमा AI एजेन्टहरू सार्दा तीन स्तम्भमा विश्वासलाग्दो ध्यान आवश्यक पर्छ:

1. **अवलोकनीयता** — एजेन्टले के गरिरहेको छ, कति समय लाग्छ, र कुन उपकरणहरू कल गर्छ भन्ने कुरामा तपाईंले दृश्यता चाहिन्छ। ट्रेसिङ र लगिङविना उत्पादन समस्याहरू डिबग गर्न लगभग असम्भव हुन्छ।

2. **मूल्यांकन** — स्वचालित गुणस्तर जाँचहरूले सुनिश्चित गर्छन् कि एजेन्टका प्रतिक्रिया समयसँगै शुद्ध, पूर्ण, र उपयोगी रहुन्। एक मूल्याङ्कन गर्ने एजेन्टले परिभाषित मापदण्डहरूसँग मिलाएर प्रतिक्रियाहरूलाई स्कोर गर्न सक्छ।

3. **लागत व्यवस्थापन** — टोकनको प्रयोगले सिधै लागतमा प्रभाव पार्छ। प्रोम्प्ट अनुकूलन, मोडेल चयन, र क्यासिङजस्ता रणनीतिहरूले गुणस्तर नघटाइ खर्च नियन्त्रणमा राख्न मद्दत गर्दछन्।


## एक निगरानीयोग्य एजेन्ट निर्माण

हामी यात्रा उपकरणहरू परिभाषित गर्छौं र विलम्बता अनुगमन गर्न एजेन्टको कललाई समय मापनले लपेट्छौं। उत्पादनमा तपाईं OpenTelemetry वा त्यस्तै कुनै ट्रेसिङ ब्याकएन्डसँग एकीकृत गर्नुहुनेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन ढाँचाहरू

सामान्य उत्पादन ढाँचा भनेको दोस्रो एजेन्टलाई **मूल्यांकनकर्ता**को रूपमा प्रयोग गर्नु हो। मूल्यांकनकर्ताले प्राथमिक एजेन्टको प्रतिक्रियालाई पूर्वनिर्धारित मापदण्डहरू जस्तै पूर्णता, सटीकता, र सहायकताको आधारमा अंकित गर्छ।

यसले सक्षम बनाउँछ:
- प्रतिक्रिया प्रयोगकर्तासम्म पुग्नु अघि स्वचालित गुणस्तर गेटहरू
- प्रम्प्टहरू वा मोडेलहरू परिवर्तन हुँदा रेग्रेशन पत्ता लगाउने
- समयसँगै एजेन्टको प्रदर्शनको निरन्तर अनुगमन


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत व्यवस्थापन रणनीतिहरू

उत्पादन AI एजेन्टहरूको लागि लागत नियन्त्रण महत्वपूर्ण छ। यहाँ प्रमुख रणनीतिहरू छन्:

| रणनीति | वर्णन |
|---|---|
| **प्रम्प्ट अनुकूलन** | सिस्टम निर्देशनहरू संक्षिप्त राख्नुहोस्। इनपुट टोकनहरू घटाउन अनावश्यक प्रसङ्ग हटाउनुहोस्। |
| **मोडेल छनोट** | वर्गीकरण वा एक्स्ट्र्याक्सन जस्ता साधा कार्यहरूको लागि साना, सस्ता मोडेलहरू (जस्तै GPT-4o-mini) प्रयोग गर्नुहोस्, र जटिल तर्कका लागि ठूला मोडेलहरू आरक्षित गर्नुहोस्। |
| **क्यासिङ** | उपकरण नतिजाहरू र बारम्बारका क्वेरीहरू क्यास गर्नुहोस् ताकि अनावश्यक API कलहरू नदोहोरियोस्। |
| **टोकन बजेट** | अप्रत्याशित रूपमा लामो उत्तरहरू रोक्न `max_tokens` सीमा सेट गर्नुहोस्। |
| **ब्याचिङ** | जहाँ सम्भव छ, बहु प्रयोगकर्ता क्वेरीहरूलाई एउटै API कलमा समूहीकरण गर्नुहोस्। |

व्यवहारमा, तहगत दृष्टिकोण राम्रो काम गर्छ: सीधा अनुरोधहरू छिटो र सस्तो मोडेलतर्फ मार्गनिर्देश गर्नुहोस् र केवल जटिल क्वेरीहरूलाई बढी सक्षम मोडेलमा उकास्नुहोस्।


## सारांश

यस पाठमा तपाईंले कसरी सिक्नुभयो:

1. **निरीक्षणक्षमता थप्नुहोस्** एजेन्ट अन्तरक्रियाहरूमा समय मापन र लगिङ्ग सहित, ट्रेसिङ र मोनिटरिङका लागि आधार तयार गर्दै।
2. **एजेन्ट प्रतिक्रियाहरूको मूल्याङ्कन गर्नुहोस्** स्वचालित रूपमा एक मूल्याङ्कन एजेन्ट प्रयोग गरेर जसले पूर्णता, शुद्धता, र उपयोगिताको स्कोर दिन्छ।
3. **लागत व्यवस्थापन गर्नुहोस्** प्रम्प्ट अनुकूलन, मोडेल चयन, क्याचिङ, र टोकन बजेटमार्फत।

यी उत्पादन सम्बन्धी ढाँचाहरूले सुनिश्चित गर्न मद्दत गर्छन् कि तपाईंका AI एजेन्टहरू ठूलो परिमाणमा भरपर्दो, मापनयोग्य, र लागत-प्रभावी छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यस दस्तावेजलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयास गर्छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा असंगतता हुन सक्दछ। मूल भाषामा रहेको दस्तावेजलाई नै आधिकारिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीका लागि पेशेवर मानव अनुवाद सिफारिस गरिन्छ। यो अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
